In [1]:
import numpy as np, os, glob                                                                                                                                              
from collections import defaultdict                                                                                                                                       
from cim_driver import CIMDriver, CIMModel, weight_to_chunks, bias_to_u32                                                                                                 
                                                                                                                                                                            
def read_hex_u8(path):                                                                                                                                                    
    with open(path) as f:                                                                                                                                                 
        return [int(line.strip(), 16) & 0xFF for line in f if line.strip()]                                                                                               
                                                                                                                                                                            
drv = CIMDriver("cim_soc.bit", use_dma=True)                                                                                                                              
d = np.load("lenet5_data/lenet5_qparams.npz")                                                                                                                             
                                                                                                                                                                            
model = CIMModel(drv)                                                                                                                                                     
model.add_conv(d["conv1_weight"], d["conv1_bias"],                                                                                                                        
                zp=int(d["conv1_zp"]), mult=int(d["conv1_mult"]),                                                                                                          
                shift=int(d["conv1_shift"]), stride=1, padding=0, relu=True)                                                                                               
model.add_pool(2, 2)                                                                                                                                                      
model.add_conv(d["conv2_weight"], d["conv2_bias"],                                                                                                                        
                zp=int(d["conv2_zp"]), mult=int(d["conv2_mult"]),                                                                                                          
                shift=int(d["conv2_shift"]), stride=1, padding=0, relu=True)                                                                                               
model.add_pool(2, 2)                                                                                                                                                      
model.add_fc(256, 120, weight_to_chunks(d["fc3_weight"]), bias_to_u32(d["fc3_bias"]),                                                                                     
            zp=int(d["fc3_zp"]), mult=int(d["fc3_mult"]), shift=int(d["fc3_shift"]), relu=True)                                                                          
model.add_fc(120, 84, weight_to_chunks(d["fc4_weight"]), bias_to_u32(d["fc4_bias"]),                                                                                      
            zp=int(d["fc4_zp"]), mult=int(d["fc4_mult"]), shift=int(d["fc4_shift"]), relu=True)                                                                          
model.add_fc(84, 10, weight_to_chunks(d["fc5_weight"]), bias_to_u32(d["fc5_bias"]),                                                                                       
            zp=int(d["fc5_zp"]), mult=int(d["fc5_mult"]), shift=int(d["fc5_shift"]), relu=False)                                                                         
                                                                                                                                                                            
img_dir = "lenet5_data/test_images"                                                                                                                                       
img_files = sorted(glob.glob(os.path.join(img_dir, "img_????.hex")))[:10]                                                                                                 
                                                                                                                                                                            
agg = defaultdict(float)                                                                                                                                                  
n = 0                                                                                                                                                                     
for path in img_files:                                                                                                                                                    
    img = np.array(read_hex_u8(path), dtype=np.uint8).reshape(1, 28, 28)                                                                                                  
    pred, logits, prof = model.predict(img, profile=True)                                                                                                                 
    n += 1                                                                                                                                                                
    for layer in prof["layers"]:                                                                                                                                          
        for k, v in layer.items():                                                                                                                                        
            if k.endswith("_ms") and k != "total_ms":                                                                                                                     
                agg[k] += v                                                                                                                                               
                                                                                                                                                                            
total_measured = sum(agg.values())                                                                                                                                        
print(f"\n{'Metric':<30} {'avg_ms':>10} {'pct':>8}")                                                                                                                      
print("-" * 48)                                                                                                                                                           
for k in sorted(agg, key=lambda x: -agg[x]):                                                                                                                              
    print(f"{k:<30} {agg[k]/n:>10.2f} {agg[k]/total_measured*100:>7.1f}%")                                                                                                
print("-" * 48)                                                                                                                                                           
print(f"{'TOTAL (components)':<30} {total_measured/n:>10.2f}")                                                                                                            
print(f"\nAvg per image: {total_measured/n:.1f} ms")                                                                                                                      
                                                                                                                                                                            
# DMA breakdown                                                                                                                                                           
print("\n=== DMA-specific (per layer type) ===")                                                                                                                          
for layer in prof["layers"]:                                                                                                                                              
    dma_keys = [k for k in layer if "dma" in k]                                                                                                                           
    if dma_keys:                                                                                                                                                          
        print(f"\n  {layer['name']} ({layer['type']}):")                                                                                                                  
        for k in dma_keys:                                                                                                                                                
            print(f"    {k}: {layer[k]:.2f} ms") 



Metric                             avg_ms      pct
------------------------------------------------
read_out_ms                        256.76    61.7%
load_x_ms                           67.71    16.3%
setup_ms                            50.75    12.2%
im2col_ms                           21.65     5.2%
compute_ms                           7.66     1.8%
dma_w_setup_ms                       3.66     0.9%
dma_w_transfer_ms                    2.90     0.7%
dma_b_transfer_ms                    1.62     0.4%
dma_x_transfer_ms                    1.61     0.4%
dma_b_setup_ms                       1.03     0.2%
dma_x_setup_ms                       0.94     0.2%
------------------------------------------------
TOTAL (components)                 416.29

Avg per image: 416.3 ms

=== DMA-specific (per layer type) ===

  conv_1x5x5_to_6 (conv):
    dma_x_setup_ms: 0.00 ms
    dma_x_transfer_ms: 0.00 ms

  conv_6x5x5_to_16 (conv):
    dma_x_setup_ms: 0.00 ms
    dma_x_transfer_ms: 0.00 ms

  fc_256x